# Unidad 5 — Notebook en Google Colab: Pipeline interactivo de clasificación

Este notebook está pensado para usarse **en clase** y permitir experimentar rápidamente con:

- **Selección de características**: color, textura (patrones), forma, o **fusión**.
- **Elección de algoritmo supervisado** (y parámetros principales).
- Impacto en el **pipeline** completo: extracción → escalado → modelo → evaluación.

## Abrir en Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navasergio/pdi-material-estudiantes/blob/main/pdi-fuente/Unidad5/Unidad5_Colab_Pipeline_Interactivo.ipynb)

> Nota: si estás usando un fork o un repo distinto, ajusta la URL del badge.


In [ ]:
# === 0) Setup (Colab) ===
# Si corres esto fuera de Colab, puedes comentar la parte de widgets.

import sys
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB =', IN_COLAB)

# Dependencias (en Colab suelen estar casi todas preinstaladas).
# Puedes comentar/ajustar según el avance de la clase.
if IN_COLAB:
    !pip -q install -U scikit-image ipywidgets

import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Widgets (interactividad)
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

import ipywidgets as widgets


## 1) Dataset: Flower Photos (rápido y visual)

Usamos el dataset clásico de flores (5 clases) para poder comparar efectos de:
- color vs textura vs forma,
- y ver cómo cambian diferentes modelos supervisados.


In [ ]:
# === 1) Descargar / cargar dataset ===
import tensorflow as tf
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'

# Descargar y extraer
path_to_zip = tf.keras.utils.get_file('flower_photos.tgz', origin=DATA_URL, extract=True)
data_root = Path(path_to_zip).parent / 'flower_photos'

# Si se extrajo bajo otra carpeta (por ejemplo, flower_photos_extracted)
if not data_root.exists() or not any(data_root.iterdir()):
    posibles_rutas = list(Path(path_to_zip).parent.glob('*flower_photos*'))
    for p in posibles_rutas:
        if p.is_dir():
            data_root = p
            break

# Si dentro de la carpeta extraída hay OTRA subcarpeta llamada 'flower_photos'
if (data_root / 'flower_photos').exists() and (data_root / 'flower_photos').is_dir():
    data_root = data_root / 'flower_photos'

print('Dataset dir:', data_root)

if not data_root.exists():
    raise FileNotFoundError(f"No se encontró la carpeta extraída en: {data_root}")

classes = sorted([p.name for p in data_root.iterdir() if p.is_dir() and p.name != 'LICENSE.txt'])
print('Clases:', classes)

# Recolectar rutas
all_paths = []
all_labels = []
for i, cls in enumerate(classes):
    for img_path in (data_root/cls).glob('*.jpg'):
        all_paths.append(str(img_path))
        all_labels.append(i)

all_paths = np.array(all_paths)
all_labels = np.array(all_labels)
print('Total imágenes:', len(all_paths))

# Vista rápida
if len(all_paths) > 0:
    idx = np.random.choice(len(all_paths), size=9, replace=False)
    plt.figure(figsize=(8,8))
    for j,k in enumerate(idx):
        img = tf.keras.utils.load_img(all_paths[k], target_size=(160,160))
        plt.subplot(3,3,j+1)
        plt.imshow(img)
        plt.title(classes[all_labels[k]])
        plt.axis('off')
    plt.tight_layout()
else:
    print("¡Error! No se encontraron imágenes en el directorio especificado.")

## 2) Parámetros del laboratorio (lo que se ajusta en clase)

La idea es que en clase modifiques estos parámetros y los alumnos vean el impacto en métricas y matriz de confusión.


In [ ]:
# === 2) Parámetros base ===
BASE_CONFIG = {
    # Para que sea rápido en clase
    'max_per_class': 120,
    'img_size': 96,

    # Features
    'feature_set': 'fusion',  # color | textura | forma | fusion
    'color_bins': 8,
    'glcm_dist': 2,
    'glcm_levels': 16,

    # Modelo
    'model': 'svm',  # logreg | svm | rf | knn
    'svm_c': 3.0,
    'svm_gamma': 'scale',
    'rf_trees': 250,
    'knn_k': 7,

    # Split
    'test_size': 0.25,
    'random_state': 42,
}
BASE_CONFIG


## 3) Funciones: cargar imágenes y extraer características

Separamos la extracción por tipo de descriptor para que sea didáctico.


In [ ]:
# === 3) Extracción de características ===
import cv2
from skimage.feature import graycomatrix, graycoprops


def load_rgb(path, img_size):
    bgr = cv2.imread(path)
    if bgr is None:
        raise ValueError(f'No se pudo leer: {path}')
    bgr = cv2.resize(bgr, (img_size, img_size), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb


def feat_color_hist(rgb, bins=8):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    # Histograma 3D H,S,V (compacto)
    hist = cv2.calcHist([hsv], [0,1,2], None, [bins,bins,bins], [0,180, 0,256, 0,256])
    hist = cv2.normalize(hist, hist).flatten()
    return hist


def feat_texture_glcm(rgb, dist=2, levels=16):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    # cuantizar para GLCM
    gray_q = np.floor(gray / (256/levels)).astype(np.uint8)
    gray_q = np.clip(gray_q, 0, levels-1)

    glcm = graycomatrix(gray_q, distances=[dist], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=levels, symmetric=True, normed=True)

    props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    feats = []
    for p in props:
        v = graycoprops(glcm, p).flatten()  # (dist x angles) -> vector
        feats.append(v)
    return np.concatenate(feats).astype(np.float32)


def feat_shape_hu(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(blur, 60, 160)

    # Contornos y Hu moments del contorno más grande
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(cnts) == 0:
        return np.zeros(7, dtype=np.float32)
    c = max(cnts, key=cv2.contourArea)
    m = cv2.moments(c)
    hu = cv2.HuMoments(m).flatten()
    # transform log para estabilidad
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    return hu.astype(np.float32)


def extract_features(paths, cfg):
    X = []
    for p in paths:
        rgb = load_rgb(p, cfg['img_size'])

        feats = []
        fs = cfg['feature_set']
        if fs in ('color', 'fusion'):
            feats.append(feat_color_hist(rgb, bins=cfg['color_bins']))
        if fs in ('textura', 'fusion'):
            feats.append(feat_texture_glcm(rgb, dist=cfg['glcm_dist'], levels=cfg['glcm_levels']))
        if fs in ('forma', 'fusion'):
            feats.append(feat_shape_hu(rgb))

        X.append(np.concatenate(feats))

    return np.vstack(X).astype(np.float32)


## 4) Construir el modelo (pipeline supervisado)

La parte clave didáctica: **mismo dataset y mismas features**, pero cambia el algoritmo y parámetros.


In [ ]:
# === 4) Modelos y pipeline ===

def build_estimator(cfg):
    model = cfg['model']

    if model == 'logreg':
        clf = LogisticRegression(max_iter=2000, n_jobs=None)
        # Recomendado escalar
        return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

    if model == 'svm':
        clf = SVC(C=float(cfg['svm_c']), gamma=cfg['svm_gamma'], kernel='rbf')
        return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

    if model == 'rf':
        clf = RandomForestClassifier(n_estimators=int(cfg['rf_trees']), random_state=cfg['random_state'], n_jobs=-1)
        return Pipeline([('clf', clf)])

    if model == 'knn':
        clf = KNeighborsClassifier(n_neighbors=int(cfg['knn_k']))
        return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

    raise ValueError(f"Modelo no soportado: {model}")


## 5) Ejecutar un experimento

Este bloque hace el flujo completo: muestreo → features → split → entrenamiento → evaluación.


In [ ]:
# === 5) Ejecutar experimento ===
from sklearn.metrics import classification_report

_feature_cache = {}


def sample_balanced(paths, labels, max_per_class, random_state=42):
    rng = np.random.default_rng(random_state)
    keep_idx = []
    for cls in np.unique(labels):
        idx = np.where(labels == cls)[0]
        k = min(max_per_class, len(idx))
        keep_idx.append(rng.choice(idx, size=k, replace=False))
    keep_idx = np.concatenate(keep_idx)
    rng.shuffle(keep_idx)
    return paths[keep_idx], labels[keep_idx]


def run_experiment(cfg):
    paths_s, y_s = sample_balanced(all_paths, all_labels, cfg['max_per_class'], cfg['random_state'])

    cache_key = (
        cfg['img_size'], cfg['feature_set'], cfg['color_bins'], cfg['glcm_dist'], cfg['glcm_levels'],
        tuple(paths_s.tolist())
    )
    if cache_key in _feature_cache:
        X = _feature_cache[cache_key]
    else:
        X = extract_features(paths_s, cfg)
        _feature_cache[cache_key] = X

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_s, test_size=cfg['test_size'], stratify=y_s, random_state=cfg['random_state']
    )

    est = build_estimator(cfg)
    est.fit(X_train, y_train)
    y_pred = est.predict(X_test)

    print('=== Config ===')
    print(cfg)
    print()

    print('=== Reporte ===')
    print(classification_report(y_test, y_pred, target_names=classes, digits=3))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    fig, ax = plt.subplots(figsize=(6.5,6.5))
    disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
    plt.tight_layout()
    plt.show()

    return est

# Prueba rápida con la config base
_ = run_experiment(dict(BASE_CONFIG))


## 6) Interactividad (en clase)

Usa los controles para cambiar features/modelo/parámetros y observar el impacto.

Sugerencia de dinámica:
1) Arranca con feature_set=fusion y model=svm.
2) Cambia a color vs textura vs forma.
3) Cambia de modelo: logreg → svm → rf → knn.
4) Ajusta parámetros (C, árboles, k) y discute overfitting/underfitting.


In [ ]:
# === 6) UI con widgets ===
feature_dd = widgets.Dropdown(options=['color','textura','forma','fusion'], value=BASE_CONFIG['feature_set'], description='Features:')
model_dd = widgets.Dropdown(options=['logreg','svm','rf','knn'], value=BASE_CONFIG['model'], description='Modelo:')

maxpc = widgets.IntSlider(value=BASE_CONFIG['max_per_class'], min=30, max=250, step=10, description='N/clase')
imgsz = widgets.IntSlider(value=BASE_CONFIG['img_size'], min=64, max=160, step=16, description='Img size')

color_bins = widgets.IntSlider(value=BASE_CONFIG['color_bins'], min=4, max=16, step=1, description='Bins')
glcm_dist = widgets.IntSlider(value=BASE_CONFIG['glcm_dist'], min=1, max=6, step=1, description='GLCM d')

svm_c = widgets.FloatLogSlider(value=BASE_CONFIG['svm_c'], base=10, min=-1, max=1.0, step=0.05, description='SVM C')
rf_trees = widgets.IntSlider(value=BASE_CONFIG['rf_trees'], min=50, max=600, step=50, description='RF trees')
knn_k = widgets.IntSlider(value=BASE_CONFIG['knn_k'], min=1, max=21, step=2, description='kNN k')

btn = widgets.Button(description='Ejecutar experimento', button_style='primary')
out = widgets.Output()

param_box = widgets.VBox([svm_c, rf_trees, knn_k])

def refresh_param_visibility(*_):
    svm_c.layout.display = 'none'
    rf_trees.layout.display = 'none'
    knn_k.layout.display = 'none'

    if model_dd.value == 'svm':
        svm_c.layout.display = ''
    elif model_dd.value == 'rf':
        rf_trees.layout.display = ''
    elif model_dd.value == 'knn':
        knn_k.layout.display = ''

refresh_param_visibility()
model_dd.observe(refresh_param_visibility, names='value')


def on_run(_):
    with out:
        out.clear_output(wait=True)
        cfg = dict(BASE_CONFIG)
        cfg['feature_set'] = feature_dd.value
        cfg['model'] = model_dd.value
        cfg['max_per_class'] = int(maxpc.value)
        cfg['img_size'] = int(imgsz.value)
        cfg['color_bins'] = int(color_bins.value)
        cfg['glcm_dist'] = int(glcm_dist.value)

        cfg['svm_c'] = float(svm_c.value)
        cfg['rf_trees'] = int(rf_trees.value)
        cfg['knn_k'] = int(knn_k.value)

        _ = run_experiment(cfg)

btn.on_click(on_run)

ui = widgets.VBox([
    widgets.HBox([feature_dd, model_dd]),
    widgets.HBox([maxpc, imgsz]),
    widgets.HBox([color_bins, glcm_dist]),
    param_box,
    btn,
    out
])
ui


---

### Ideas para extender en clase

- Agregar PCA antes del modelo y observar su impacto.
- Cambiar la estrategia de forma (por ejemplo, segmentación más robusta) y comparar Hu moments.
- Variar el split y observar variabilidad.
- Reemplazar el dataset por uno propio del curso.
